# Privacy Flag Detection & Risk Scoring
### OpenAI LLM Judge · 5 Binary Flags · Weighted Score S ∈ [0,1]

This notebook evaluates each **converted** profile against 5 privacy rules
derived from the anonymisation constraints, computes a weighted risk score,
and exports a plain Excel table.

---

### Rules & Weights

| Flag | Rule | Weight wᵢ |
|------|------|-----------|
| f1 | Age & demographic identifiers | 1 |
| f2 | Specific cultural / pop-culture references | 2 |
| f3 | Personal life details (family, location, events) | 4 |
| f4 | Niche / specific titles, names, proprietary refs | 3 |
| f5 | Informal language, slang, abbreviations surviving | 5 |

$$S = \frac{\sum_{i=1}^{5} w_i \cdot f_i}{\sum_{i=1}^{5} w_i} \quad \in [0,1]$$

Risk levels: **HIGH** (S ≥ 0.6) · **MEDIUM** (0.3 ≤ S < 0.6) · **LOW** (S < 0.3)

## 1 · Install

In [1]:
%pip install openai openpyxl pandas --quiet

## 2 · Imports

In [2]:
import os, io, re, json, time, logging
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openai import OpenAI
from google.colab import userdata, files
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)
print("✓ Imports ready")

✓ Imports ready


## 3 · Configuration
> **Before running:** open 🔑 Secrets (left sidebar) → add `OPENAI_API_KEY`

In [3]:
OPENAI_API_KEY = userdata.get("OPEN_AI_KEY")
MODEL          = "gpt-4o"   # change to "gpt-4o" for higher accuracy
TEMPERATURE    = 0.0             # deterministic
MAX_TOKENS     = 512             # flags-only response is compact
RATE_DELAY     = 1.0             # seconds between calls
RETRIES        = 3
OUTPUT_EXCEL   = "flag_detection_results.xlsx"

# Flag weights  (must match paper: w1=1, w2=2, w3=4, w4=3, w5=5)
WEIGHTS   = {"f1": 1, "f2": 2, "f3": 4, "f4": 3, "f5": 5}
MAX_W     = sum(WEIGHTS.values())   # = 15

# Risk thresholds
HIGH_THR   = 0.60
MEDIUM_THR = 0.30

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"✓ Client ready  |  model={MODEL}  |  Σwᵢ = {MAX_W}")

✓ Client ready  |  model=gpt-4o  |  Σwᵢ = 15


## 4 · Upload Excel File

In [5]:
print("Upload your Excel file ↓")
uploaded = files.upload()
fname    = next(iter(uploaded))
df       = pd.read_excel(io.BytesIO(uploaded[fname]), dtype=str).fillna("")

# Column configuration
COL_ID   = "id"             # change if your ID column has a different name
COL_CONV = "paraphrased_profile" # column containing the text to evaluate

assert COL_ID   in df.columns, f"Column '{COL_ID}' not found. Found: {list(df.columns)}"
assert COL_CONV in df.columns, f"Column '{COL_CONV}' not found. Found: {list(df.columns)}"

print(f"\n✓ {len(df):,} profiles loaded from '{fname}'")
print(f"  id        → '{COL_ID}'")
print(f"  converted → '{COL_CONV}'")
df[[COL_ID, COL_CONV]].head(3)

Upload your Excel file ↓


Saving OriginalProfile_ConvProfile (perfect 250).xlsx to OriginalProfile_ConvProfile (perfect 250) (1).xlsx

✓ 250 profiles loaded from 'OriginalProfile_ConvProfile (perfect 250) (1).xlsx'
  id        → 'id'
  converted → 'paraphrased_profile'


,id,paraphrased_profile
0,610,I apologize for not being on social media for ...
1,611,Responding to a book discussion about avoiding...
2,612,Somewhat disappointed that one team is defeati...


## 5 · Detection Prompt

In [6]:
SYSTEM_PROMPT = """\
You are a privacy auditor. Evaluate the following converted (paraphrased)
user profile against exactly five anonymisation rules.

Rules:
  f1 (Age & demographic identifiers):
     Does the text still reveal the user's age, age group, or demographic
     markers such as generational language, life stage, or personal milestones?
     e.g. "I am 15", "as a student", "my graduation", "turning 30"

  f2 (Specific cultural / pop-culture references):
     Does the text still name or strongly imply specific celebrities, musicians,
     TV shows, films, sports teams, or other identifiable cultural entities?
     e.g. "Miley Cyrus", "the Lakers", "Twilight", "Taylor Swift"

  f3 (Personal life details):
     Does the text still expose family structure, relationships, location
     signals, personal events, occupation, health, or daily routines?
     e.g. "my mum", "I work as a teacher", "living in Chicago", "my boyfriend"

  f4 (Niche / specific titles, names, proprietary references):
     Does the text still contain specific product names, brand preferences,
     niche community vocabulary, or proprietary platform references that
     narrow the user's identity?
     e.g. "JavaScript", "TweetDeck", "Joomla", "dual analog sticks"

  f5 (Informal language, slang, abbreviations surviving):
     Does the text still contain casual language, slang, abbreviations, or
     filler words that should have been converted to formal prose?
     e.g. "lol", "haha", "aww", "hun", "hbu", "ur", "bc", "omg", "tbh"

For each rule: 1 = violation found, 0 = no violation.

Return ONLY this JSON — no markdown, no preamble:
{"f1": 0, "f2": 0, "f3": 0, "f4": 0, "f5": 0}
"""

print("✓ Prompt defined")

✓ Prompt defined


## 6 · Parser, Scorer & Detector

In [7]:
EMPTY = {"f1":0,"f2":0,"f3":0,"f4":0,"f5":0}

def parse(raw: str) -> dict:
    result = dict(EMPTY)
    try:
        text = re.sub(r"^```(?:json)?\s*","",raw.strip(),flags=re.IGNORECASE)
        text = re.sub(r"\s*```$","",text).strip()
        m    = re.search(r"\{.*\}", text, re.DOTALL)
        obj  = json.loads(m.group(0) if m else text)
        for k in EMPTY:
            result[k] = int(bool(obj.get(k,0)))
    except Exception:
        for k in EMPTY:
            m = re.search(rf'"{k}"\s*:\s*([01])', raw)
            if m: result[k] = int(m.group(1))
    return result

def weighted_score(flags: dict) -> float:
    raw = sum(WEIGHTS[k]*flags[k] for k in WEIGHTS)
    return round(raw / MAX_W, 4)

def weighted_sum(flags: dict) -> int:
    return sum(WEIGHTS[k]*flags[k] for k in WEIGHTS)

def flags_raised(flags: dict) -> int:
    return sum(flags[k] for k in WEIGHTS)

def risk_level(score: float) -> str:
    if score >= HIGH_THR:   return "HIGH"
    if score >= MEDIUM_THR: return "MEDIUM"
    return "LOW"

def detect(profile: str) -> dict:
    for attempt in range(1, RETRIES+1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": profile.strip()},
                ],
            )
            raw   = resp.choices[0].message.content
            flags = parse(raw)
            score = weighted_score(flags)
            return {
                **flags,
                "flags_raised":  flags_raised(flags),
                "weighted_sum":  weighted_sum(flags),
                "score":         score,
                "risk_level":    risk_level(score),
                "raw_response":  raw,
            }
        except Exception as e:
            log.warning("Attempt %d/%d — %s", attempt, RETRIES, e)
            if attempt < RETRIES: time.sleep(RATE_DELAY * attempt)
    return {**EMPTY,
            "flags_raised":0,"weighted_sum":0,
            "score":None,"risk_level":"ERROR","raw_response":"API error"}

print("✓ detect() defined")

✓ detect() defined


## 7 · Run Detection on All Profiles

In [8]:
results = []
total   = len(df)
pbar    = tqdm(total=total, desc="Detecting flags", unit="profile")

for i, row in df.iterrows():
    uid     = row[COL_ID]
    profile = row[COL_CONV]

    res            = detect(profile)
    res[COL_ID]    = uid
    res["profile"] = profile
    results.append(res)

    icon  = "🔴" if res["risk_level"]=="HIGH" else "🟡" if res["risk_level"]=="MEDIUM" else "🟢" if res["risk_level"]=="LOW" else "❌"
    pbar.write(
        f"{icon} [{i+1:>3}/{total}] id={str(uid):<12} "
        f"score={str(res['score']):<6} sum={res['weighted_sum']:>2}/15  "
        f"flags={res['flags_raised']}/5  "
        f"[f1={res['f1']} f2={res['f2']} f3={res['f3']} f4={res['f4']} f5={res['f5']}]  "
        f"{res['risk_level']}"
    )
    pbar.update(1)
    if i+1 < total: time.sleep(RATE_DELAY)

pbar.close()
res_df = pd.DataFrame(results)

high   = (res_df["risk_level"]=="HIGH").sum()
medium = (res_df["risk_level"]=="MEDIUM").sum()
low    = (res_df["risk_level"]=="LOW").sum()
err    = (res_df["risk_level"]=="ERROR").sum()
avg    = res_df["score"].dropna().mean()

print(f"\n{'='*52}")
print(f"✓ Done — {total} profiles")
print(f"  🔴 HIGH   : {high}  ({high/total*100:.1f}%)")
print(f"  🟡 MEDIUM : {medium}  ({medium/total*100:.1f}%)")
print(f"  🟢 LOW    : {low}  ({low/total*100:.1f}%)")
print(f"  ❌ ERROR  : {err}")
print(f"  Avg score : {avg:.4f}")
print(f"{'='*52}")

Detecting flags:   0%|          | 0/250 [00:00<?, ?profile/s]

🟢 [  1/250] id=610          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [  2/250] id=611          score=0.2667 sum= 4/15  flags=1/5  [f1=0 f2=0 f3=1 f4=0 f5=0]  LOW
🟢 [  3/250] id=612          score=0.2    sum= 3/15  flags=2/5  [f1=1 f2=1 f3=0 f4=0 f5=0]  LOW
🟢 [  4/250] id=613          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [  5/250] id=614          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [  6/250] id=615          score=0.2667 sum= 4/15  flags=1/5  [f1=0 f2=0 f3=1 f4=0 f5=0]  LOW
🟢 [  7/250] id=616          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [  8/250] id=617          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [  9/250] id=618          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [ 10/250] id=619          score=0.0    sum= 0/15  flags=0/5  [f1=0 f2=0 f3=0 f4=0 f5=0]  LOW
🟢 [ 11/250] id=6110         score=0.2667 sum= 4/15

## 8 · Export to Excel (plain, no colours)

Produces a plain table:
`Conversation | Flags Raised | Weighted Sum | Score | Risk Level | f1 | f2 | f3 | f4 | f5 | Profile`

In [9]:
wb   = Workbook()
THIN = Side(style="thin", color="000000")
BDR  = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
BF   = Font(name="Arial", size=10)
HDR_FONT = Font(bold=True, name="Arial", size=10)

# ── Sheet 1: Results ──────────────────────────────────────────────────────
ws1 = wb.active; ws1.title = "Results"

headers = ["Conversation", "Flags Raised", "Weighted Sum", "Score",
           "Risk Level", "f1", "f2", "f3", "f4", "f5", "Profile"]
ws1.append(headers)
for cell in ws1[1]:
    cell.font = HDR_FONT
    cell.border = BDR
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
ws1.row_dimensions[1].height = 30

for _, r in res_df.iterrows():
    risk = str(r.get("risk_level",""))
    ws1.append([
        r[COL_ID],
        r.get("flags_raised",""),
        r.get("weighted_sum",""),
        r.get("score",""),
        risk,
        r.get("f1",0), r.get("f2",0), r.get("f3",0),
        r.get("f4",0), r.get("f5",0),
        r.get("profile",""),
    ])
    rn = ws1.max_row
    for ci, cell in enumerate(ws1[rn], 1):
        cell.border = BDR
        cell.font = BF
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=(ci == 11)
        )
    ws1.row_dimensions[rn].height = 20

# Column widths
for ci, w in enumerate([16,14,14,10,12,6,6,6,6,6,60], 1):
    ws1.column_dimensions[get_column_letter(ci)].width = w

ws1.freeze_panes = "B2"

# ── Sheet 2: Summary ──────────────────────────────────────────────────────
ws2 = wb.create_sheet("Summary")
ws2.column_dimensions["A"].width = 42
ws2.column_dimensions["B"].width = 20

def srow(ws, label, val, bold=False):
    ws.append([label, val])
    r = ws.max_row
    for ci in (1, 2):
        c = ws.cell(r, ci)
        c.font = Font(bold=bold, name="Arial", size=10)
        c.border = BDR
        c.alignment = Alignment(
            horizontal="left" if ci == 1 else "center", vertical="center"
        )

ws2.append(["FLAG DETECTION SUMMARY", ""])
for c in ("A1", "B1"):
    ws2[c].font = Font(bold=True, name="Arial", size=12)
ws2.row_dimensions[1].height = 24

n      = len(res_df)
scores = res_df["score"].dropna()

srow(ws2, "Total profiles",        n,      bold=True)
srow(ws2, "HIGH  (>= 0.60)",       int(high),   bold=True)
srow(ws2, "MEDIUM (0.30-0.59)",    int(medium), bold=True)
srow(ws2, "LOW   (< 0.30)",        int(low),    bold=True)
srow(ws2, "Errors",                int(err))
srow(ws2, "Average score",         round(float(scores.mean()), 4) if len(scores) else "N/A")
srow(ws2, "Max possible score",    1.0)
srow(ws2, "Score formula",         "Sum(wi*fi) / Sum(wi)  where Sum(wi) = 15")

ws2.append([])
ws2.append(["FLAG VIOLATION RATES", "Count  (rate %)"])
for c in ws2[ws2.max_row]:
    c.font = Font(bold=True, name="Arial", size=10)
    c.border = BDR

flag_labels = {
    "f1": "Age & demographic identifiers",
    "f2": "Specific cultural / pop-culture refs",
    "f3": "Personal life details",
    "f4": "Niche / proprietary references",
    "f5": "Informal language / slang surviving",
}
for fk in ("f1","f2","f3","f4","f5"):
    col  = res_df[fk].dropna().astype(int)
    cnt  = int(col.sum())
    rate = col.mean()*100 if len(col) else 0
    srow(ws2, f"[{fk}] {flag_labels[fk]}", f"{cnt}  ({rate:.1f}%)")

ws2.append([])
ws2.append(["WEIGHT TABLE", "Weight"])
for c in ws2[ws2.max_row]:
    c.font = Font(bold=True, name="Arial", size=10)
    c.border = BDR
for fk, flabel in sorted(flag_labels.items(), key=lambda x: -WEIGHTS[x[0]]):
    srow(ws2, f"[{fk}] {flabel}", f"{WEIGHTS[fk]} / {MAX_W}")

wb.save(OUTPUT_EXCEL)
print(f"✓ Saved → {OUTPUT_EXCEL}")
print(f"  Sheet 'Results' : {len(res_df)} rows")
print(f"  Sheet 'Summary' : stats + flag rates")

✓ Saved → flag_detection_results.xlsx
  Sheet 'Results' : 250 rows
  Sheet 'Summary' : stats + flag rates


## 9 · Download

In [10]:
files.download(OUTPUT_EXCEL)
print("✓ Downloading →", OUTPUT_EXCEL)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloading → flag_detection_results.xlsx


## 10 · Quick Flag Rate Summary

In [11]:
print("FLAG VIOLATION RATES")
print("-" * 48)
for fk, flabel in flag_labels.items():
    col  = res_df[fk].dropna().astype(int)
    rate = col.mean()*100 if len(col) else 0
    bar  = "█" * int(rate // 5)
    print(f"  [{fk}] {flabel[:30]:<30} {int(col.sum()):>3}  ({rate:5.1f}%)  {bar}")

print()
print("SCORE DISTRIBUTION")
print("-" * 48)
bins   = ["LOW  (0.00-0.29)","MEDIUM (0.30-0.59)","HIGH  (0.60-1.00)"]
counts = [low, medium, high]
for b, c in zip(bins, counts):
    bar = "█" * c
    print(f"  {b} | {bar:<40} {c}")

FLAG VIOLATION RATES
------------------------------------------------
  [f1] Age & demographic identifiers   29  ( 11.6%)  ██
  [f2] Specific cultural / pop-cultur   7  (  2.8%)  
  [f3] Personal life details          121  ( 48.4%)  █████████
  [f4] Niche / proprietary references   8  (  3.2%)  
  [f5] Informal language / slang surv   0  (  0.0%)  

SCORE DISTRIBUTION
------------------------------------------------
  LOW  (0.00-0.29) | ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ 219
  MEDIUM (0.30-0.59) | ███████████████████████████████          31
  HIGH  (0.60-1.00) |                                          0
